# Healthy-Comparison Dataset Check: Chula-RBC-12-Dataset

**This is a standalone notebook** - it doesn't depend on any cells from
`01_setup_and_inspect.ipynb`, so there's nothing to copy-paste or keep in
sync between notebooks. Just run this one top to bottom.

**Project framing:** part of an experimental research/screening tool, not a
medical diagnostic device.

**What this notebook does:** downloads a candidate healthy-comparison
dataset (Chula-RBC-12-Dataset, which has an explicit "Normal cell" label -
something our other datasets are missing) and inspects its structure and
label format, so we can decide whether and how to use it. **It does not
train anything or extract "Normal cell" crops yet** - this is a pause point
before that.

**No GPU needed for this notebook** - it's just downloading and inspecting
data, so you can leave the Colab runtime on CPU and save your GPU quota for
actual training later (`Runtime > Change runtime type > CPU` if it's
currently set to GPU).


## 1. Get the project code

**Why:** so downloaded data lands in the same `data/` folder structure the
rest of the project uses.


In [ ]:
import os

REPO_URL = "https://github.com/aural0i/Sickle-cell-detection"
BRANCH = "claude/sickle-cell-cnn-research-g6fipc"
if not os.path.isdir("/content/Sickle-cell-detection"):
    !git clone --branch {BRANCH} {REPO_URL} /content/Sickle-cell-detection
else:
    !git -C /content/Sickle-cell-detection pull
%cd /content/Sickle-cell-detection


## 2. About this dataset

**Chula-RBC-12-Dataset** (Naruenatthanaset et al., "Red Blood Cell
Segmentation with Overlapping Cell Separation and Classification on
Imbalanced Dataset," arXiv:2012.01321, 2021), hosted on Zenodo
(https://zenodo.org/records/5638201).

Per the dataset's own page: 706 whole blood-smear images (640x480) with
over 20,000 individually labeled red blood cells across 12 shape classes,
where **class 0 is explicitly "Normal cell"** - a real healthy-comparison
label. Small download (~58 MB).

**License - needs your eyes on one detail.** The dataset's GitHub repo
(`Chula-PIC-Lab/Chula-RBC-12-Dataset`) carries an MIT License (permissive:
use/modify/redistribute, just keep the copyright+license notice) - Claude
confirmed this directly from the repo. But Zenodo lists the *dataset's own*
license separately as `"Other (Open)"`, and Claude cannot load the Zenodo
page itself to check whether that means the same MIT terms or something
else (`zenodo.org` is blocked from Claude's development environment).
**Please open https://zenodo.org/records/5638201 yourself and copy back
exactly what the "License" section says.**

**Citation required if used:** Naruenatthanaset et al., arXiv:2012.01321
(2021) - we'll credit this in the project's README/references regardless of
the final licensing decision.


In [ ]:
import requests

os.makedirs("data/healthy_comparison", exist_ok=True)

CHULA_URL = "https://zenodo.org/records/5638201/files/Chula-PIC-Lab/Chula-RBC-12-Dataset-dataset.zip?download=1"
out_path = "data/healthy_comparison/Chula-RBC-12-Dataset-dataset.zip"

print("Downloading Chula-RBC-12-Dataset (~58 MB)...")
resp = requests.get(CHULA_URL, stream=True, timeout=60)
resp.raise_for_status()
with open(out_path, "wb") as out:
    for chunk in resp.iter_content(chunk_size=8192):
        out.write(chunk)
print("Download complete:", os.path.getsize(out_path) / 1e6, "MB")


In [ ]:
import zipfile

with zipfile.ZipFile(out_path) as z:
    z.extractall("data/healthy_comparison")

print("Extracted. Top-level contents of data/healthy_comparison/:")
print(os.listdir("data/healthy_comparison"))


## 3. Inspect folder structure, file types, and label format

**Why:** these are whole-smear images with per-cell coordinate labels, not
pre-cropped single-cell images like our other datasets, and the README
didn't fully specify the label file format. We need to see it directly
before writing any extraction code.


In [ ]:
def describe_tree(root, max_depth=3, max_files_per_dir=5):
    root = os.path.abspath(root)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        if depth > max_depth:
            dirnames[:] = []
            continue
        indent = "  " * depth
        print(f"{indent}{os.path.basename(dirpath) or dirpath}/  ({len(filenames)} files, {len(dirnames)} subfolders)")
        for fn in sorted(filenames)[:max_files_per_dir]:
            print(f"{indent}  - {fn}")
        if len(filenames) > max_files_per_dir:
            print(f"{indent}  ... ({len(filenames) - max_files_per_dir} more files)")

print("=" * 60)
print("HEALTHY-COMPARISON CANDIDATE: data/healthy_comparison/")
print("=" * 60)
describe_tree("data/healthy_comparison")


In [ ]:
from collections import Counter

ext_counter = Counter()
for dirpath, dirnames, filenames in os.walk("data/healthy_comparison"):
    for f in filenames:
        ext_counter[os.path.splitext(f)[1].lower()] += 1

print("File extensions found:")
for ext, count in ext_counter.most_common():
    print(f"  {ext or '(no extension)'}: {count}")


In [ ]:
# Print the full contents of a few non-image files (likely the label/annotation
# files) so we can see the exact format the README didn't fully specify.
import glob

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
non_image_exts = {ext for ext in ext_counter if ext and ext not in image_exts}
print("Non-image file extensions:", non_image_exts)
print()

shown = 0
for dirpath, dirnames, filenames in os.walk("data/healthy_comparison"):
    for f in sorted(filenames):
        ext = os.path.splitext(f)[1].lower()
        if ext in non_image_exts and shown < 5:
            full = os.path.join(dirpath, f)
            size = os.path.getsize(full)
            print("=" * 60)
            print(full, f"({size} bytes)")
            print("=" * 60)
            if size < 5000:
                with open(full, "r", errors="replace") as fh:
                    print(fh.read())
            else:
                print("(file is large - showing first 1000 characters)")
                with open(full, "r", errors="replace") as fh:
                    print(fh.read(1000))
            print()
            shown += 1


In [ ]:
# Do image files and label files share a naming pattern? (e.g. 001.jpg + 001.txt)
img_files = sorted(glob.glob("data/healthy_comparison/**/*.jpg", recursive=True) +
                    glob.glob("data/healthy_comparison/**/*.png", recursive=True))
print(f"Total image files found: {len(img_files)}")
print()
print("First 5 images and any same-named files next to them:")
for p in img_files[:5]:
    base = os.path.splitext(p)[0]
    matches = [m for m in glob.glob(base + ".*") if m != p]
    print(" ", p, "-> other files with same name:", matches)


In [ ]:
# Look for any bundled README/LICENSE/citation files inside the download itself
print("Documentation/license-like files found inside data/healthy_comparison/:")
for pattern in ["*README*", "*readme*", "*LICENSE*", "*license*", "*CITATION*", "*.md"]:
    for p in glob.glob(f"data/healthy_comparison/**/{pattern}", recursive=True):
        print(" ", p)


## 4. Report back before we go further

Copy back to Claude:
- The file-extension counts and the printed label-file contents (so we can
  nail down the exact annotation format and how to pull out just the
  "Normal cell" / class-0 entries)
- The image/label naming-correspondence check
- Whatever the Zenodo page's "License" section actually says, in full

**Nothing has been extracted or used for training yet** - this notebook only
downloaded and described the data. Once we can see the real label format,
Claude will write the extraction code to pull out just the Normal-cell
crops/coordinates, count how many there are, and report back whether it's a
large enough sample before anything gets wired into the evaluation pipeline.
